# OC Transpo route-time bus allocation (proof of concept)

This notebook loads the `Mar 23 Subset OC Transpo.csv` dataset, builds a compact integer optimization model, and solves for the number of buses assigned to each route and time block.

Model choices for this draft:
- decision variable `x[r,t]` = integer buses assigned to route `r` in time block `t`
- bounds come directly from the CSV (`Min Buses`, `Max Buses`)
- objective maximizes a simple **effective benefit** proxy
- constraints enforce a **fleet cap by time block** and a **daily operating budget**

This is meant to be concise and workable as a starting point rather than a final calibrated model.


In [20]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# Try local folder first, then /mnt/data as fallback
candidate_paths = [
    Path('Mar 23 Subset OC Transpo.csv'),
    Path('data/Mar 23 Subset OC Transpo.csv'),
    Path('/mnt/data/Mar 23 Subset OC Transpo.csv'),
]

csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError('Could not find Mar 23 Subset OC Transpo.csv')

print(f'Using data file: {csv_path}')
df = pd.read_csv(csv_path)
df.head()


Using data file: data\Mar 23 Subset OC Transpo.csv


,Route,Time Block,Ridership,Route Length (km),Round Trip Time (hr),Cost per Bus ($/hr),Min Buses,Max Buses,Benefit per Bus
0,5,Early AM,23,12,1.16,120,1,6,1.15
1,5,AM Peak,292,12,1.16,120,2,12,14.60
2,5,Midday,566,12,1.16,120,2,14,28.30
3,5,PM Peak,303,12,1.16,120,2,12,15.20
4,5,Evening,162,12,1.16,120,1,8,8.10


In [21]:
# Assumptions for the proof-of-concept model
# These are easy to change later.

time_block_hours = {
    'Early AM': 4,
    'AM Peak': 3,
    'Midday': 6,
    'PM Peak': 3,
    'Evening': 5,
    'Night': 3,
}

fleet_cap = {
    'Early AM': 12,
    'AM Peak': 22,
    'Midday': 24,
    'PM Peak': 22,
    'Evening': 16,
    'Night': 10,
}

# Define key parameter

# Maximum number of buses that can be assigned to any single route-time combination, to prevent over-concentration.
fleet_cap_per_block = 40

# Daily budget for this subset only
budget = 50000

# Preprocess
model_df = df.copy()
model_df['Block Hours'] = model_df['Time Block'].map(time_block_hours)

# Effective benefit: reward routes with higher demand, while discounting very long round trips.
model_df['Effective Benefit'] = model_df['Benefit per Bus'] / model_df['Round Trip Time (hr)']

# Daily cost of assigning one bus to that route-time combination
model_df['Unit Cost'] = model_df['Cost per Bus ($/hr)'] * model_df['Block Hours']

model_df[['Route','Time Block','Ridership','Min Buses','Max Buses','Block Hours','Unit Cost','Effective Benefit']].head(10)


,Route,Time Block,Ridership,Min Buses,Max Buses,Block Hours,Unit Cost,Effective Benefit
0,5,Early AM,23,1,6,4,480,0.99
1,5,AM Peak,292,2,12,3,360,12.59
2,5,Midday,566,2,14,6,720,24.40
3,5,PM Peak,303,2,12,3,360,13.10
4,5,Evening,162,1,8,5,600,6.98
5,5,Night,73,1,6,3,360,3.15
6,6,Early AM,177,2,10,4,480,5.40
7,6,AM Peak,1724,4,20,3,360,52.56
8,6,Midday,3513,4,24,6,720,107.13
9,6,PM Peak,2781,4,22,3,360,84.82


In [ ]:
# Solve the integer program with SciPy MILP if available.
# Minimize negative benefit = maximize benefit.

try:
    from scipy.optimize import milp, LinearConstraint, Bounds
except Exception as e:
    raise ImportError(
        'This notebook currently expects scipy.optimize.milp to be available. '
        'If needed, we can swap this to PuLP or Gurobi later.'
    ) from e

n = len(model_df)
c = -model_df['Effective Benefit'].to_numpy(dtype=float)  # maximize benefit
integrality = np.ones(n, dtype=int)
lb = model_df['Min Buses'].to_numpy(dtype=float)
ub = model_df['Max Buses'].to_numpy(dtype=float)
bounds = Bounds(lb, ub)



# --- Feasibility diagnostics ---
print('\n=== Feasibility diagnostics ===')

min_by_block = model_df.groupby('Time Block')['Min Buses'].sum()
max_by_block = model_df.groupby('Time Block')['Max Buses'].sum()

print('\nMin buses required by time block:')
print(min_by_block)

print('\nMax buses allowed by time block:')
print(max_by_block)

print(f'\nFleet cap per block = {fleet_cap_per_block}')
for tb, val in min_by_block.items():
    if val > fleet_cap_per_block:
        print(f'WARNING: infeasible at {tb}: min required {val} > fleet cap {fleet_cap_per_block}')

min_cost = (model_df['Min Buses'] * model_df['Unit Cost']).sum()
print(f'\nCost of minimum-service solution = {min_cost:,.2f}')
print(f'Budget = {budget:,.2f}')
if min_cost > budget:
    print('WARNING: infeasible because minimum-service cost exceeds budget')

A = []
lower = []
upper = []

# Fleet cap by time block
for tb, cap in fleet_cap.items():
    row = (model_df['Time Block'] == tb).astype(float).to_numpy()
    A.append(row)
    lower.append(-np.inf)
    upper.append(cap)

# Daily budget
A.append(model_df['Unit Cost'].to_numpy(dtype=float))
lower.append(-np.inf)
upper.append(budget)

constraints = LinearConstraint(np.vstack(A), np.array(lower), np.array(upper))

res = milp(c=c, constraints=constraints, integrality=integrality, bounds=bounds)

print('Solver success:', res.success)
print('Status code   :', res.status)
print('Message       :', res.message)
if not res.success:
    raise RuntimeError('MILP solve failed; inspect the assumptions or solver availability.')



=== Feasibility diagnostics ===

Min buses required by time block:
Time Block
AM Peak     15
Early AM     7
Evening      8
Midday      15
Night        5
PM Peak     15
Name: Min Buses, dtype: int64

Max buses allowed by time block:
Time Block
AM Peak     76
Early AM    36
Evening     52
Midday      90
Night       40
PM Peak     82
Name: Max Buses, dtype: int64

Fleet cap per block = 40

Cost of minimum-service solution = 31,560.00
Budget = 50,000.00
Solver success: True
Status code   : 0
Message       : Optimization terminated successfully. (HiGHS Status 7: Optimal)


In [23]:
# Attach solution back to the dataframe
solution = np.rint(res.x).astype(int)
model_df['Optimal Buses'] = solution
model_df['Benefit Contribution'] = model_df['Effective Benefit'] * model_df['Optimal Buses']
model_df['Cost Contribution'] = model_df['Unit Cost'] * model_df['Optimal Buses']

solution_df = model_df[[
    'Route','Time Block','Ridership','Min Buses','Max Buses','Optimal Buses',
    'Unit Cost','Effective Benefit','Benefit Contribution','Cost Contribution'
]].copy()

solution_df.sort_values(['Time Block','Route'])


,Route,Time Block,Ridership,Min Buses,Max Buses,Optimal Buses,Unit Cost,Effective Benefit,Benefit Contribution,Cost Contribution
1,5,AM Peak,292,2,12,2,360,12.59,25.17,720
7,6,AM Peak,1724,4,20,4,360,52.56,210.24,1440
13,7,AM Peak,1649,4,20,11,360,55.71,612.80,3960
19,9,AM Peak,383,2,10,2,360,19.15,38.30,720
25,10,AM Peak,879,3,14,3,360,33.30,99.89,1080
0,5,Early AM,23,1,6,1,480,0.99,0.99,480
6,6,Early AM,177,2,10,2,480,5.40,10.79,960
12,7,Early AM,110,2,8,2,480,3.72,7.43,960
18,9,Early AM,44,1,6,1,480,2.20,2.20,480
24,10,Early AM,42,1,6,1,480,1.59,1.59,480


In [24]:
# Summary checks
summary = {
    'Objective (proxy benefit)': model_df['Benefit Contribution'].sum(),
    'Total cost': model_df['Cost Contribution'].sum(),
    'Budget': budget,
    'Slack in budget': budget - model_df['Cost Contribution'].sum(),
}

for tb in fleet_cap:
    used = model_df.loc[model_df['Time Block'] == tb, 'Optimal Buses'].sum()
    summary[f'Fleet used - {tb}'] = used
    summary[f'Fleet cap - {tb}'] = fleet_cap[tb]

pd.Series(summary)


Objective (proxy benefit)    5,442.72
Total cost                  49,680.00
Budget                      50,000.00
Slack in budget                320.00
Fleet used - Early AM            7.00
Fleet cap - Early AM            12.00
Fleet used - AM Peak            22.00
Fleet cap - AM Peak             22.00
Fleet used - Midday             24.00
Fleet cap - Midday              24.00
Fleet used - PM Peak            22.00
Fleet cap - PM Peak             22.00
Fleet used - Evening            16.00
Fleet cap - Evening             16.00
Fleet used - Night              10.00
Fleet cap - Night               10.00
dtype: float64

In [25]:
# Compact route-time table for easy copy/paste into the report
pivot = solution_df.pivot(index='Route', columns='Time Block', values='Optimal Buses')
pivot = pivot[['Early AM','AM Peak','Midday','PM Peak','Evening','Night']]
pivot


Time Block,Early AM,AM Peak,Midday,PM Peak,Evening,Night
Route,,,,,,
5,1,2,2,2,1,1
6,2,4,4,4,2,1
7,2,11,13,11,10,6
9,1,2,2,2,1,1
10,1,3,3,3,2,1


## Notes for next iteration

Natural upgrades when you want more realism:
1. replace the proxy objective with waiting-time or ridership-response logic
2. convert the fleet constraint to depend explicitly on round-trip time and dispatch rate
3. compare LP relaxation vs integer solve
4. add a cut-generation experiment for your project writeup
5. move from this 5-route subset to more of the GTFS / ridership data
